# 00. Test the AS4777 curve keystone

This notebook validates "shared/as4777_curves.py" **offline** (no AWS, no data).

**What it checks**
1. The module imports and reads set-points from `ciccada_config.AS4777`.
2. The Volt-VAr, Volt-Watt and capability curves hit their known set-points.
3. The Python functions and the SQL-string generators agree (single source of truth).

In [ ]:
import sys
from pathlib import Path

# 00_test_curves.ipynb is two levels below bms_sa_review.
BMS_ROOT = Path("../..").resolve()

if str(BMS_ROOT) not in sys.path:
    sys.path.insert(0, str(BMS_ROOT))

from shared import as4777_curves as c
from shared.ciccada_config import AS4777

print("Package root:", BMS_ROOT)
print("Imported OK. Set-points:")
print("  VVAR:", AS4777["VVAR"])
print("  VW:  ", AS4777["VW"])
print("  TOL: ", AS4777["TOL_FRAC"])

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

## 1. Volt-VAr required-Q curve. Set points check

Expected (for S_rated = 5 kW):
- V ≤ 207 → +2.20 kvar (full supply, +0.44×5)
- V = 213.5 (midpoint) → +1.10 (half)
- V = 220~240 (deadband) → 0
- V = 249 (midpoint of 240-258) → -1.50 (half of -0.60×5)
- V ≥ 258 → -3.00 (full absorb)

In [ ]:
import math

S = 5.0

vvar_checks = {
    200.0:  2.2,
    207.0:  2.2,
    213.5:  1.1,
    220.0:  0.0,
    240.0:  0.0,
    249.0: -1.5,
    258.0: -3.0,
    265.0: -3.0,
}

for v, expected in vvar_checks.items():
    actual = c.vvar_required_q(v, S)

    assert math.isclose(actual, expected, abs_tol=1e-12), (
        f"V={v}: expected {expected}, got {actual}"
    )

    print(
        f"V={v:6.1f} -> "
        f"Q_required={actual:+7.3f} kvar"
    )

print("Volt-VAr set-point checks passed.")

In [ ]:
# Plot the full curve
V = np.linspace(200, 270, 400)
Q = [c.vvar_required_q(v, S) for v in V]
plt.figure(figsize=(8, 4))
plt.plot(V, Q, lw=2)
plt.axhline(0, color='k', lw=0.5)
for vx in (207, 220, 240, 258):
    plt.axvline(vx, ls=':', color='grey', lw=0.8)
plt.xlabel("Voltage (V)"); plt.ylabel("Required Q (kvar)")
plt.title("Volt-VAr required-Q curve (Australia A, S_rated = 5 kW)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 2. Volt-Watt max-P curve

Expected (S_rated = 5 kW): 100% (5.0) at/below 253 V, ramp to 20% (1.0) at 260 V.

In [ ]:
vw_checks = {
    250.0: 5.0,
    253.0: 5.0,
    256.5: 3.0,
    260.0: 1.0,
    263.0: 1.0,
}

for v, expected in vw_checks.items():
    actual = c.vw_max_p(v, S)

    assert math.isclose(actual, expected, abs_tol=1e-12), (
        f"V={v}: expected {expected}, got {actual}"
    )

    print(
        f"V={v:6.1f} -> "
        f"P_max={actual:6.3f} kW"
    )

print("Volt-Watt set-point checks passed.")

## 3. Minimum reactive-power capability and conformance floor

`q_cap_absorbing()` represents the AS/NZS 4777.2:2020 Figure 2.1
minimum absorbing-capability boundary at the observed active power.

Below 20% of rated apparent power, zero means that the standard provides
no quantified minimum reactive-power capability. It does not mean that
the inverter is physically incapable of producing reactive power.

Above 80%, the fixed-P apparent-power circle falls toward zero as P
approaches S_rated. For field conformance, reactive-power priority may
require active-power reduction. The separate
`q_conformance_floor_absorbing()` function represents that assessment
floor.

In [ ]:
capability_checks = {
    0.50: 0.0,
    0.75: 0.0,
    1.00: -2.2,
    2.00: -2.2,
    3.00: -2.2,
    3.50: -2.625,
    4.00: -3.0,
    4.50: -math.sqrt(4.75),
    5.00: 0.0,
    5.50: 0.0,
}

conformance_checks = {
    0.50: 0.0,
    0.75: 0.0,
    1.00: -2.2,
    2.00: -2.2,
    3.00: -2.2,
    3.50: -2.625,
    4.00: -3.0,
    4.50: -3.0,
    5.00: -3.0,
    5.50: -3.0,
}

print("P       Figure 2.1 boundary    Conformance floor")

for p in capability_checks:
    capability = c.q_cap_absorbing(p, S)
    conformance = c.q_conformance_floor_absorbing(p, S)

    assert math.isclose(
        capability,
        capability_checks[p],
        abs_tol=1e-12,
    ), (
        f"P={p}: incorrect Figure 2.1 boundary: "
        f"expected {capability_checks[p]}, got {capability}"
    )

    assert math.isclose(
        conformance,
        conformance_checks[p],
        abs_tol=1e-12,
    ), (
        f"P={p}: incorrect conformance floor: "
        f"expected {conformance_checks[p]}, got {conformance}"
    )

    print(
        f"{p:4.2f}    "
        f"{capability:+9.3f} kvar       "
        f"{conformance:+9.3f} kvar"
    )

print("Capability and conformance-floor checks passed.")

## 4. Measurement tolerance

In [ ]:
AC_CAPACITY = 5.0

# Read the configured tolerance from shared/ciccada_config.py.
TOL_FRAC = AS4777["TOL_FRAC"]
TOL_EXPECTED = TOL_FRAC * AC_CAPACITY

# Confirm that the configured value still matches the intended standard value.
assert math.isclose(
    TOL_FRAC,
    0.04,
    abs_tol=1e-12,
), f"Expected TOL_FRAC=0.04, got {TOL_FRAC}"

# Check the Python tolerance helper.
assert math.isclose(
    c.add_tol_kw(0.0, AC_CAPACITY, sign=+1),
    +TOL_EXPECTED,
    abs_tol=1e-12,
)

assert math.isclose(
    c.add_tol_kw(0.0, AC_CAPACITY, sign=-1),
    -TOL_EXPECTED,
    abs_tol=1e-12,
)

assert math.isclose(
    c.add_tol_kw(-3.0, AC_CAPACITY, sign=+1),
    -3.0 + TOL_EXPECTED,
    abs_tol=1e-12,
)

assert math.isclose(
    c.add_tol_kw(-3.0, AC_CAPACITY, sign=-1),
    -3.0 - TOL_EXPECTED,
    abs_tol=1e-12,
)

# Check the generated SQL.
expected_tol_sql = f"{TOL_FRAC} * ac_capacity_kw"

assert (
    c.tol_kw_sql("ac_capacity_kw")
    == expected_tol_sql
), (
    f"Expected SQL '{expected_tol_sql}', "
    f"got '{c.tol_kw_sql('ac_capacity_kw')}'"
)

print(f"Configured tolerance fraction: {TOL_FRAC:.2%}")
print(
    f"Tolerance for {AC_CAPACITY:.1f} kW nameplate: "
    f"±{TOL_EXPECTED:.2f} kW/kvar"
)
print("Tolerance checks passed.")

## 5. SQL-generator inspection

The generated SQL should contain the same configured set-points and
piecewise branches as the Python functions.

This offline section verifies that the correct rated-power proxy is passed
to each generator and displays the generated expressions for inspection.
Numerical execution of the SQL occurs in the Athena pipeline validations.

In [ ]:
print("Volt-VAr SQL fragment:")
vvar_sql = c.vvar_required_q_sql("V", "ac_capacity_kw")
print(vvar_sql)

print("\nVolt-Watt SQL fragment:")
vw_sql = c.vw_max_p_sql("V", "ac_capacity_kw")
print(vw_sql)

print("\nFigure 2.1 capability SQL fragment:")
q_cap_sql = c.q_cap_absorbing_sql(
    "P_kW",
    "ac_capacity_kw",
)
print(q_cap_sql)

print("\nReactive-priority conformance-floor SQL fragment:")
q_floor_sql = c.q_conformance_floor_absorbing_sql(
    "P_kW",
    "ac_capacity_kw",
)
print(q_floor_sql)

assert "s_99" not in q_cap_sql.lower()
assert "s_99" not in q_floor_sql.lower()
assert "ac_capacity_kw" in q_cap_sql
assert "ac_capacity_kw" in q_floor_sql

print("\nSQL generators use the documented nameplate proxy.")

## Done

If every set-point above matches expectation, the keystone is validated. 

Proceed to `01_run_stage1_pipeline.ipynb`.